# Demonstration of the visit sequence archive

## Preparation of the environment

### User setup

In addition to the standard dependencies of `rubin_sim`, to additional packages need to be in the environment to run this notebook:
- `postgresql` needs to be installed in the kernel's environment. It can be installed with `conda install postgresql`.
- `testing-postgresql` needs to be installed from pip: `pip install testing-postgresql` . (It isn't available in conda.)

### Imports

In [1]:
import pickle
from pathlib import Path
from tempfile import TemporaryDirectory
from datetime import datetime, timedelta, timezone

from astropy.time import Time
import numpy as np
import pandas as pd
import sqlite3
from lsst.resources import ResourcePath

import rubin_scheduler
from rubin_sim import maf
from rubin_sim.sim_archive import vseqarchive
from rubin_sim.sim_archive.tempdb import LocalOnlyPostgresql
from rubin_scheduler.scheduler import sim_runner
from rubin_scheduler.scheduler.model_observatory import ModelObservatory
from rubin_scheduler.scheduler.example import example_scheduler
from rubin_scheduler.scheduler.utils import SchemaConverter

### Create a temporary sandbox archive

To avoid either requiring access to the production archive or metadata database, or alteration of the production archive or alteration metadata database in the course of the demonstation, create a local archive and metadata database to work with.

This will be deleted at the end of the notebook.

Create a temporary directory to be a common root for the archive and metadata database. This will be deleted later, so everything well be cleaned up.

In [2]:
demo_dir = TemporaryDirectory()

Create a directory for the data archive, the location where we will store the actual files, including the table of visits.
Access will be managed through `lsst.resources.ResourcePath`, so the `ResourcePath` to the root of the archive is what we will ultiamtely end up with.

In [3]:
archive_url = "file://" + demo_dir.name + "/archive/"
archive = ResourcePath(archive_url)

Make a postgresql database in another subdirectory of our demonstration base, and take a look at the connection parameters.

In [4]:
md_database = LocalOnlyPostgresql(base_dir=demo_dir.name)
metadata_db_kwargs = md_database.psycopg2_dsn()
metadata_db_kwargs


{'port': 62422,
 'host': '/var/folders/_x/wlrrv2nn0mj2y1jdkmphj2fm0000j6/T/tmp7mrsmtn7/tmp',
 'user': 'neilsen',
 'database': 'test'}

Create an instance of `vseqarchive.VisitSequenceArchiveMetadata` to be our interface to it, and use this interface to create the schema in it what we will use to hold the metadata.

In [5]:
md_db_schema = 'testvseqmd'
archive_metadata =  vseqarchive.VisitSequenceArchiveMetadata(
    metadata_db_kwargs=metadata_db_kwargs,
    metadata_db_schema=md_db_schema,
)
archive_metadata.create_schema_in_database()

Created test database and schema  testvseqmd


Delete the instance of `VisitSequencArchiveMetadata` we just used to create the schema (`archive_metadata`).
We *could* just reuse this later, but I want to start the next section by creating a new instance, so lets get clean up this old one.

In [6]:
del(archive_metadata)

## Create the interface we will use to talk to the metadata database

If we didn't just delete it, we could have continued using the previous one, but I want a clean beginning here.
Lets looks at the connection parameters we need to use:

In [7]:
metadata_db_kwargs

{'port': 62422,
 'host': '/var/folders/_x/wlrrv2nn0mj2y1jdkmphj2fm0000j6/T/tmp7mrsmtn7/tmp',
 'user': 'neilsen',
 'database': 'test'}

We are using a local-only demonstration database here, so the `'host'` is a directory name, but in production this will be the host name for the production database.

In [8]:
archive_metadata =  vseqarchive.VisitSequenceArchiveMetadata(
    metadata_db_kwargs=metadata_db_kwargs,
    metadata_db_schema=md_db_schema,
)

At this point we can directly query the database, but it isn't very interesting yet:

In [9]:
archive_metadata.query("SELECT * FROM simulations")

[]

But, we can already do things like check out what tables are in our schema:

In [10]:
archive_metadata.query("SELECT table_name FROM information_schema.tables WHERE table_schema='testvseqmd'")

[('visitseq',),
 ('simulations',),
 ('completed',),
 ('mixedvisitseq',),
 ('tags',),
 ('comments',),
 ('files',),
 ('simulations_extra',),
 ('conda_env',),
 ('conda_packages',),
 ('simulation_packages',),
 ('nightly_stats',),
 ('maf_metrics',),
 ('maf_summary_metrics',),
 ('maf_metric_sets',),
 ('maf_summary',),
 ('maf_healpix_stats',)]

## Configuring a shell for use of the command line tool

To use the command line tool, the following environment variables are used
to point to the correct instance of the metadata archive database: `VSARCHIVE_PGDATABASE`, `VSARCHIVE_PGHOST`, `VSARCHIVE_PGUSER` and `VSARCHIVE_PGSCHEMA`. Alternately, these can be specified using the `--database`, `--host`, `--user` option to the `vseqarchive` command, but setting them every time is cumbersome.

In addition, `PGPASSFILE` must point to the (readable only by user) credentials file with the password.
For this running test database, the set of bash commands to set up the environment can be contsructed thus:

In [11]:
print(f"export VSARCHIVE_PGDATABASE={metadata_db_kwargs['database']}")
print(f"export VSARCHIVE_PGHOST={metadata_db_kwargs['host']}")
print(f"export VSARCHIVE_PGPORT={metadata_db_kwargs['port']}")
print(f"export VSARCHIVE_PGUSER={metadata_db_kwargs['user']}")
print(f"export VSARCHIVE_PGSCHEMA={md_db_schema}")

export VSARCHIVE_PGDATABASE=test
export VSARCHIVE_PGHOST=/var/folders/_x/wlrrv2nn0mj2y1jdkmphj2fm0000j6/T/tmp7mrsmtn7/tmp
export VSARCHIVE_PGPORT=62422
export VSARCHIVE_PGUSER=neilsen
export VSARCHIVE_PGSCHEMA=testvseqmd


Most python commands for interacting with the archive shown below have corresponding shell commands. With `rubin_sim` installed, get a list with `vseqarchive --help`.

## Adding an entry for visits queried from consdb

`sample_consdb.db` is an opsim database that is the result of querying the consdb using `sv_survey.simulate_sv.fetch_previous_sv_visits`, and then truncated it to 100 visits to avoid taking too much space for this demonstration. Let's add an entry for that to the metadata database.

In [12]:
consdb_visits_db = "sample_consdb.db"
with sqlite3.connect(consdb_visits_db) as conn:
    consdb_visits = pd.DataFrame(maf.get_sim_data(conn))
consdb_visits.head()

,observationId,exposure_name,controller,day_obs,seq_num,physical_filter,band,fieldRA,fieldDec,rotSkyPos,...,sunRA,sunDec,zero_point_1s,zero_point_1s_pred,cloud_extinction,skyBrightness,fiveSigmaDepth,nexp,night,note
0,2025062000248,MC_O_20250620_000248,O,20250620,248,i_39,i,259.031417,-16.841287,124.088653,...,89.647094,23.435576,28.276500,28.257699,-0.018801,20.173639,23.796964,1,0.0,"pair_33, iz, a"
1,2025062000249,MC_O_20250620_000249,O,20250620,249,i_39,i,255.235319,-13.581529,109.375195,...,89.647718,23.435578,28.252555,28.257004,0.004449,20.159062,23.775703,1,0.0,"pair_33, iz, a"
2,2025062000250,MC_O_20250620_000250,O,20250620,250,i_39,i,252.234921,-13.082034,100.507091,...,89.648264,23.435579,28.246745,28.256957,0.010211,20.170747,23.767636,1,0.0,"pair_33, iz, a"
3,2025062000251,MC_O_20250620_000251,O,20250620,251,i_39,i,249.206911,-12.511995,91.779497,...,89.648807,23.435580,28.250320,28.256660,0.006341,20.192683,23.818806,1,0.0,"pair_33, iz, a"
4,2025062000252,MC_O_20250620_000252,O,20250620,252,i_39,i,246.144506,-11.839659,83.714383,...,89.649340,23.435581,28.252067,28.256086,0.004019,20.218659,23.935251,1,0.0,"pair_33, iz, a"


Now, record the visit metadata in the metadata database.
This bare-bones version just saves a label and a hash of the table of visits, and nothing else.
We could, if we had it, also save the string that was the query sent, and the time the query was sent.

In [13]:
sample_consdb_uuid = archive_metadata.record_completed_metadata(
    visits=consdb_visits,
    label="Sample query from consdb #1",
    first_day_obs=consdb_visits.day_obs.min(),
    last_day_obs=consdb_visits.day_obs.max()
)
sample_consdb_uuid

UUID('46d13ef4-9723-48bc-9947-600dd6e81ed1')

The `first_day_obs` and `last_day_obs` keyword arguments are optional, but recommended. These dates can be provided is integers, ISO-8601 strings, or instances of python's `datetime.date`.

It returns a unique identifier that the database can use to track this set of visits.
We can use this, for example, to take a look at the metadata we just added:

In [14]:
archive_metadata.get_visitseq_metadata(sample_consdb_uuid, 'completed')

visitseq_uuid                   46d13ef4-9723-48bc-9947-600dd6e81ed1
visitseq_sha256    [b'\x14', b'e', b'\x9b', b'H', b'\x0f', b'\x88...
visitseq_label                           Sample query from consdb #1
visitseq_url                                                    None
telescope                                                    simonyi
first_day_obs                                             2025-06-20
last_day_obs                                              2025-06-20
creation_time                       2025-09-23 21:27:11.626887+00:00
query                                                           None
Name: 0, dtype: object

We can take a look at thewhole table that holds metadata on sequences of completed visits:

In [15]:
archive_metadata.pd_read_sql(
    f"SELECT * FROM testvseqmd.completed",
)

,visitseq_uuid,visitseq_sha256,visitseq_label,visitseq_url,telescope,first_day_obs,last_day_obs,creation_time,query
0,46d13ef4-9723-48bc-9947-600dd6e81ed1,"[b'\x14', b'e', b'\x9b', b'H', b'\x0f', b'\x88...",Sample query from consdb #1,None,simonyi,2025-06-20,2025-06-20,2025-09-23 21:27:11.626887+00:00,None


## Running a simulation and adding it to the archive

Configure the scheduler and model observatory so the survey starts at the first visit in the consdb query.

In [16]:
first_consdb_dayobs = np.min(consdb_visits.day_obs)
first_consdb_datetime = (
    datetime.strptime(str(first_consdb_dayobs), "%Y%m%d").replace(tzinfo=timezone.utc) + timedelta(hours=12)
)
first_consdb_time = Time(first_consdb_datetime)
survey_start_mjd = first_consdb_time.mjd

Begin by creating instances of the `ModelObservatory` and the scheduler we want.
For this notebook, we'll use the example scheduler.

In [17]:
model_observatory = ModelObservatory(mjd_start=survey_start_mjd, downtimes='ideal')
scheduler = example_scheduler(mjd_start=survey_start_mjd)

/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/surveys/base_survey.py:570: FutureWarning: setting dither to bool deprecated, swapping to dither='night' (blob_long, gr)
  warnings.warn(
/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/surveys/base_survey.py:570: FutureWarning: setting dither to bool deprecated, swapping to dither='night' (blob_long, ri)
  warnings.warn(
/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/surveys/base_survey.py:570: FutureWarning: setting dither to bool deprecated, swapping to dither='night' (blob_long, iz)
  warnings.warn(


Optimizing ELAISS1
Optimizing XMM_LSS
Optimizing ECDFS
Optimizing COSMOS
Optimizing EDFS_a


/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/surveys/base_survey.py:570: FutureWarning: setting dither to bool deprecated, swapping to dither='night' (greedy)
  warnings.warn(
/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/surveys/base_survey.py:570: FutureWarning: setting dither to bool deprecated, swapping to dither='night' (twilight_near_sun)
  warnings.warn(
/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/surveys/base_survey.py:570: FutureWarning: setting dither to bool deprecated, swapping to dither='night' (twilight_near_sun)
  warnings.warn(
/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/surveys/base_survey.py:570: FutureWarning: setting dither to bool deprecated, swapping to dither='night' (twilight_near_sun)
  warnings.warn(
/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/surveys/base_survey.py:570: FutureWarning: setting dither to bool deprecated, swapping to dither='night' (pair_33, ug)
  warning

Convert the visits from the consdb into a form that can be added to the scheduler, and load them in:

In [18]:
consdb_obs = SchemaConverter().opsimdf2obs(consdb_visits)
scheduler.add_observations_array(consdb_obs)

/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/utils/utils.py:637: UserWarning: Column flush_by_mjd not found.
  warnings.warn(f"Column {key} not found.")
/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/utils/utils.py:637: UserWarning: Column filter not found.
  warnings.warn(f"Column {key} not found.")
/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/utils/utils.py:637: UserWarning: Column clouds not found.
  warnings.warn(f"Column {key} not found.")
/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/utils/utils.py:637: UserWarning: Column target_id not found.
  warnings.warn(f"Column {key} not found.")


Let's start our simulation the night after the last visit in our consdb query results. 

In [19]:
last_consdb_dayobs = np.max(consdb_visits.day_obs)
last_consdb_datetime = (
    datetime.strptime(str(last_consdb_dayobs), "%Y%m%d").replace(tzinfo=timezone.utc) + timedelta(hours=12)
)
last_consdb_time = Time(last_consdb_datetime)
sim_start_mjd = last_consdb_time.mjd + 1
print(f"Last visit in consdb started at {np.max(consdb_visits.obs_start)} on {last_consdb_time}, MJD {last_consdb_time.mjd}")
print(f"So the simulation should start on MJD {sim_start_mjd}, or {Time(sim_start_mjd, format='mjd').iso}")

Last visit in consdb started at 2025-06-21T04:46:15.556000 on 2025-06-20 12:00:00, MJD 60846.5
So the simulation should start on MJD 60847.5, or 2025-06-21 12:00:00.000


Construct a file name in which to save the result of our simulation:

In [20]:
sim_opsim_fname = str(Path(demo_dir.name) / 'sim1_opsim.db')
sim_opsim_fname

'/var/folders/_x/wlrrv2nn0mj2y1jdkmphj2fm0000j6/T/tmp7mrsmtn7/sim1_opsim.db'

Build a keyword args dictionary to send to `sim_runner`, and call it:

In [21]:
sim_runner_kwargs = {
    'sim_start_mjd': sim_start_mjd,
    'sim_duration': 3,
    'filename': sim_opsim_fname,
    'verbose': True,
}
final_model_observatory, final_scheduler, simulated_obs = sim_runner(
    model_observatory,
    scheduler,
    **sim_runner_kwargs
)

progress = 14.98%

/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/skybrightness_pre/sky_model_pre.py:345: UserWarning: Sun high, using bright sky approx
  warnings.warn("Sun high, using bright sky approx")
/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/model_observatory/model_observatory.py:681: UserWarning: No finite rotSkyPos value, using rotSkyPos_desired
  warnings.warn("No finite rotSkyPos value, using rotSkyPos_desired")
/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/scheduler/features/features.py:509: UserWarning: Time must flow forwards to track the feature in NObservationsCurrentSeason.Not updating season_map.
  warnings.warn(


progress = 81.66%

/Users/neilsen/devel/rubin_scheduler/rubin_scheduler/skybrightness_pre/sky_model_pre.py:359: UserWarning: Requested MJD between sunrise and sunset, returning closest maps
  warnings.warn("Requested MJD between sunrise and sunset, returning closest maps")


progress = 148.34%Skipped 0 observations
Flushed 0 observations from queue for being stale
Completed 2682 observations
ran in 0 min = 0.0 hours
Writing results to  /var/folders/_x/wlrrv2nn0mj2y1jdkmphj2fm0000j6/T/tmp7mrsmtn7/sim1_opsim.db


Now record the simulation in the metadata database.
Note that this does *not* actually save the visits themselves in the archive.

In [22]:
# Get the visits into the right data structure (a visits pandas.DataFrame)
simulated_visits = SchemaConverter().obs2opsim(simulated_obs)

# Actually add the metadata:
simulation_uuid = archive_metadata.record_simulation_metadata(
    visits=simulated_visits,
    label="Sample simulated visits 1",
    first_day_obs=Time(sim_start_mjd, format='mjd').datetime.date(),
    last_day_obs=Time(sim_start_mjd + sim_runner_kwargs['sim_duration'], format='mjd').datetime.date(),
    sim_runner_kwargs=sim_runner_kwargs,
    parent_visitseq_uuid=sample_consdb_uuid,
    parent_last_day_obs=last_consdb_datetime.date().isoformat()
)
simulation_uuid

UUID('dfa5e287-4a42-4443-b79b-fe78584f39f2')

In [23]:
archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations')

visitseq_uuid                        dfa5e287-4a42-4443-b79b-fe78584f39f2
visitseq_sha256         [b'\xd6', b'\x81', b'E', b'\x1c', b'\x83', b'1...
visitseq_label                                  Sample simulated visits 1
visitseq_url                                                         None
telescope                                                         simonyi
first_day_obs                                                  2025-06-21
last_day_obs                                                   2025-06-24
creation_time                            2025-09-23 21:27:51.691148+00:00
scheduler_version                                                    None
config_url                                                           None
conda_env_sha256                                                     None
parent_visitseq_uuid                 46d13ef4-9723-48bc-9947-600dd6e81ed1
sim_runner_kwargs       {'verbose': True, 'filename': '/var/folders/_x...
parent_last_day_obs                   

We can also update our metadata after the fact.
For example, in the above, we didn't set the `scheduler_version`.
Do it now:

In [24]:
archive_metadata.update_visitseq_metadata(simulation_uuid, 'scheduler_version', rubin_scheduler.__version__)
archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations')

visitseq_uuid                        dfa5e287-4a42-4443-b79b-fe78584f39f2
visitseq_sha256         [b'\xd6', b'\x81', b'E', b'\x1c', b'\x83', b'1...
visitseq_label                                  Sample simulated visits 1
visitseq_url                                                         None
telescope                                                         simonyi
first_day_obs                                                  2025-06-21
last_day_obs                                                   2025-06-24
creation_time                            2025-09-23 21:27:51.691148+00:00
scheduler_version                                                  3.15.0
config_url                                                           None
conda_env_sha256                                                     None
parent_visitseq_uuid                 46d13ef4-9723-48bc-9947-600dd6e81ed1
sim_runner_kwargs       {'verbose': True, 'filename': '/var/folders/_x...
parent_last_day_obs                   

## Getting extended metadata

The metadata database also stores some metadata outside the primary `simulations` table. One way of getting this extra metadata in single `pandas.Sequence` by querying the `simulations_extra` view:

In [25]:
archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations_extra')

sim_creation_day_obs                                           2025-09-23
daily_id                                                                1
visitseq_uuid                        dfa5e287-4a42-4443-b79b-fe78584f39f2
visitseq_label                                  Sample simulated visits 1
visitseq_url                                                         None
telescope                                                         simonyi
first_day_obs                                                  2025-06-21
last_day_obs                                                   2025-06-24
creation_time                            2025-09-23 21:27:51.691148+00:00
scheduler_version                                                  3.15.0
config_url                                                           None
sim_runner_kwargs       {'verbose': True, 'filename': '/var/folders/_x...
conda_env_sha256                                                     None
parent_visitseq_uuid                 4

Note that the `tags`, `comments`, and `files` fields are empty, because we have not added any to the metadata database. Lets add some tags and comments:

In [26]:
archive_metadata.tag(simulation_uuid, 'example', 'other_tag')
archive_metadata.comment(simulation_uuid, "This is a first comment.")
archive_metadata.comment(simulation_uuid, "This is a second comment.")

Now ask for the extra metadata again, this time putting it into the local `sim_extra_metadata` variable.

In [27]:
sim_extra_metadata = archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations_extra')

Now we can see our tags:

In [28]:
sim_extra_metadata['tags']

['example', 'other_tag']

and comments:

In [29]:
sim_extra_metadata['comments']

{'2025-09-23T21:27:51.714908-05:00': 'This is a first comment.',
 '2025-09-23T21:27:51.715379-05:00': 'This is a second comment.'}

We could also just get our commends in a `pandas.DataFrame` directly:

In [30]:
archive_metadata.get_comments(simulation_uuid)

,visitseq_uuid,comment_time,author,comment
0,dfa5e287-4a42-4443-b79b-fe78584f39f2,2025-09-24 02:27:51.714908+00:00,None,This is a first comment.
1,dfa5e287-4a42-4443-b79b-fe78584f39f2,2025-09-24 02:27:51.715379+00:00,None,This is a second comment.


## Save the visits themselves into the archive

The above example only records the metadata for the simulation and consdb output in the metadata database, it does not save the visits themselves into the archive.

The visits and other files can be added to an archive (based at a `lsst.resources.ResourcePath`) using `vseqarchive.add_file`.
Recall that our simulation above saved its output into a file named by the `sim_opsim_fname` above, and passed as the `filename` keyword argument to `sim_runner`. Lets add that file to the archive:

In [31]:
visits_resource_path = vseqarchive.add_file(
    vsarch=archive_metadata,
    uuid=simulation_uuid,
    origin=sim_opsim_fname,
    file_type='visits',
    archive_base=archive
)
visits_resource_path

ResourcePath("file:///var/folders/_x/wlrrv2nn0mj2y1jdkmphj2fm0000j6/T/tmp7mrsmtn7/archive/simonyi/2025-09-23/dfa5e287-4a42-4443-b79b-fe78584f39f2/visits.h5")

(Recall that we set the `archive` variable at the top of this notebook to `ResourcePath` pointing to a temporary directory that served as a sandbox for this notebook. In production it would point to the S3 bucket that holds the archived data.)

Technically, the `filetype` can be any string, so the metadata database can support new types of files without code modifications. However, possible `filetypes` should be standardized by convertion so that they are useful in practice.

The `"visits"` `filetype` is special, designating the table of visits themselves.

Now, if we look at the `simulations` entry in the metadata database, the `visitseq_url` row is set to the URL where the visits themselves can be downloaded:

In [32]:
archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations')

visitseq_uuid                        dfa5e287-4a42-4443-b79b-fe78584f39f2
visitseq_sha256         [b'\xd6', b'\x81', b'E', b'\x1c', b'\x83', b'1...
visitseq_label                                  Sample simulated visits 1
visitseq_url            file:///var/folders/_x/wlrrv2nn0mj2y1jdkmphj2f...
telescope                                                         simonyi
first_day_obs                                                  2025-06-21
last_day_obs                                                   2025-06-24
creation_time                            2025-09-23 21:27:51.691148+00:00
scheduler_version                                                  3.15.0
config_url                                                           None
conda_env_sha256                                                     None
parent_visitseq_uuid                 46d13ef4-9723-48bc-9947-600dd6e81ed1
sim_runner_kwargs       {'verbose': True, 'filename': '/var/folders/_x...
parent_last_day_obs                   

To get the file of visits, use `lsst.resources.ResourcePath` directly.
For example, if we want to copy it from the archive into `my_local_visits.h5` it our local sandbox (so it gets cleaned up), we could do this:

In [33]:
my_local_visits_path = str(Path(demo_dir.name) / 'sim1_opsim.h5')
destination_visits_rp = ResourcePath(my_local_visits_path)
origin_visits_rp = ResourcePath(archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations')['visitseq_url'])
destination_visits_rp.transfer_from(origin_visits_rp, 'copy')

In this example, the `ResourcePath` URL begins with a `"file"` URL scheme and there is no hostname, because the sandbox archive is on local disk. The production archive with have an `S3` scheme and a hostname.

No we can load our visits:

In [34]:
retrieved_visits = pd.read_hdf(my_local_visits_path, 'observations')
retrieved_visits.head()

,observationId,fieldRA,fieldDec,observationStartMJD,flush_by_mjd,visitExposureTime,band,filter,rotSkyPos,rotSkyPos_desired,...,sunDec,moonRA,moonDec,moonDistance,solarElong,moonPhase,cummTelAz,observation_reason,science_program,cloud_extinction
0,0,209.823425,-42.524808,60847.950785,60847.993042,29.2,i,i,246.487640,-113.512360,...,23.435253,38.112337,19.479523,155.914525,126.880217,27.024233,126.281742,pairs_iz_15.0,None,0.0
1,1,210.524975,-45.284589,60847.951232,60847.993042,29.2,i,i,250.790120,-109.209880,...,23.435252,38.120758,19.481415,153.440989,127.285847,27.019935,131.668097,pairs_iz_15.0,None,0.0
2,2,211.287593,-48.049133,60847.951678,60847.993042,29.2,i,i,254.503766,-105.496234,...,23.435250,38.129165,19.483305,150.901007,127.580530,27.015645,136.519726,pairs_iz_15.0,None,0.0
3,3,208.080846,-52.678853,60847.952147,60847.993042,29.2,i,i,265.894748,-94.105252,...,23.435249,38.138004,19.485292,145.898258,125.027044,27.011134,146.866062,pairs_iz_15.0,None,0.0
4,4,209.502995,-58.261585,60847.952599,60847.993042,29.2,i,i,270.673605,-89.326395,...,23.435247,38.146542,19.487211,140.713176,124.914975,27.006777,153.724641,pairs_iz_15.0,None,0.0


If you need the visits in the `sqlite3` format, you can convert with `vseqarchive.hdf5_to_opsimdb`:

In [35]:
my_local_opsimdb_path = str(Path(demo_dir.name) / 'returned_sim1_opsim.db')
vseqarchive.hdf5_to_opsimdb(my_local_visits_path, my_local_opsimdb_path)

'/var/folders/_x/wlrrv2nn0mj2y1jdkmphj2fm0000j6/T/tmp7mrsmtn7/returned_sim1_opsim.db'

In [36]:
db_retrieved_obs = maf.get_sim_data(my_local_opsimdb_path)
db_retrieved_obs[:5]

rec.array([(0, 209.82342452, -42.5248076 , 60847.95078504, 60847.99304213, 29.2, 'i', 'i', 246.48763954, -113.51236046, 2, 1.11056515, 0.52372841, 0.76302501, 0.67920656, 19.1481833 , 1, 120.18782821, 33.6, 27.98761123, 23.50688302, 64.14808873, 126.53267239, 289.30507116, -70.60343895, 0.125, -55.36086437, -12.40917539, 'pair_15, iz, a', 'lowdust', 2, 181.9155865 , -47.09107849, -47.38229963, 260.68141208, 290.06869426, 90.4835329 , 23.43525322, 38.11233673, 19.47952335, 155.91452454, 126.8802165 , 27.02423302, 126.28174232, 'pairs_iz_15.0', 'None', 0.),
           (1, 210.52497531, -45.28458946, 60847.95123206, 60847.99304213, 29.2, 'i', 'i', 250.79012006, -109.20987994, 2, 1.12076836, 0.52372841, 0.76722344, 0.68265767, 19.21144711, 1,   5.02212218, 33.6,  2.80565516, 23.5305764 , 62.76747426, 131.95248435, 293.7738549 , -66.12836112, 0.125, -55.49173326, -12.53947912, 'pair_15, iz, a', 'lowdust', 3, 182.07695261, -46.91848117, -47.38229963, 260.57818429, 289.99836439, 90.48399794, 

## Saving other files

Saving the visits (`file_type = "visits"`) is special, and has its own dedicated row in the tables of visit sequences.
The metadata database and archive support arbitrary other types of files.

For example, we can save a pickle of the schedule we used.

Begin by saving the pickle of the scheduler to a local file:

In [37]:
my_local_scheduler_pickle_path = str(Path(demo_dir.name) / 'scheduler.p')
with open(my_local_scheduler_pickle_path, 'wb') as scheduler_pickle_io:
    pickle.dump(scheduler, scheduler_pickle_io)

Now we can add it to the archive, much like we did with the visits:

In [38]:
scheduler_resource_path = vseqarchive.add_file(
    vsarch=archive_metadata,
    uuid=simulation_uuid,
    origin=my_local_scheduler_pickle_path,
    file_type='scheduler',
    archive_base=archive
)
scheduler_resource_path

ResourcePath("file:///var/folders/_x/wlrrv2nn0mj2y1jdkmphj2fm0000j6/T/tmp7mrsmtn7/archive/simonyi/2025-09-23/dfa5e287-4a42-4443-b79b-fe78584f39f2/scheduler.p")

To find the URL later, we can either query the `files` table in the metadata database directly:

In [39]:
archive_metadata.query(
    f"SELECT file_url FROM files WHERE file_type='scheduler' AND visitseq_uuid='{simulation_uuid}'"
)

[('file:///var/folders/_x/wlrrv2nn0mj2y1jdkmphj2fm0000j6/T/tmp7mrsmtn7/archive/simonyi/2025-09-23/dfa5e287-4a42-4443-b79b-fe78584f39f2/scheduler.p',)]

or get it from the `simulations_extra` view:

In [40]:
sim_extra_metadata = archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations_extra')
sim_extra_metadata['files']

{'scheduler': 'file:///var/folders/_x/wlrrv2nn0mj2y1jdkmphj2fm0000j6/T/tmp7mrsmtn7/archive/simonyi/2025-09-23/dfa5e287-4a42-4443-b79b-fe78584f39f2/scheduler.p'}

## Computing statistics by night and saving them in the metadata database

Statistics can be saved in the metadata database so they can be queried without needing to download the whole table of visits

First, compute the desired statistics thus:

In [41]:
nightly_stats = vseqarchive.compute_nightly_stats(
    simulated_visits.assign(obs_start_mjd=simulated_visits['observationStartMJD']),
    columns = ['observationStartMJD', 'azimuth', 'altitude', 'rotTelPos', 'slewDistance', 'fiveSigmaDepth']
)
nightly_stats.head()

/Users/neilsen/devel/rubin_sim/rubin_sim/sim_archive/vseqarchive.py:1490: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(level=0)


,day_obs,value_name,count,mean,std,min,p05,q1,median,q3,p95,max,accumulated
0,2025-06-21,altitude,1063.0,47.069133,12.390993,22.552886,27.013137,37.943459,46.205349,56.211939,67.802678,76.192664,False
1,2025-06-21,azimuth,1063.0,205.032185,72.475388,0.478060,34.057074,182.036899,211.791488,242.564316,329.137291,358.108652,False
2,2025-06-21,fiveSigmaDepth,1063.0,23.662358,0.889316,20.601645,21.316253,23.365865,23.862662,24.256195,24.637688,24.836510,False
3,2025-06-21,observationStartMJD,1063.0,60848.199416,0.142563,60847.950785,60847.977288,60848.076808,60848.199759,60848.321699,60848.422234,60848.445850,False
4,2025-06-21,rotTelPos,1063.0,-44.062627,9.455674,-78.505054,-64.067769,-47.194495,-43.600436,-38.484309,-31.982576,-4.686711,False


Note the `.assign(obs_start_mjd=simulated_visits['observationStartMJD']),` in the argument that takes the `pd.DataFrame` of visits. This is needed because `compute_nightly_stats` uses `obs_start_mjd` to determine the on which night (dayobs) each visit falls, and `obs_start_mjd` is the time in the schema used by consdb. The simulations do not (yet) use this schema, but this function was written in anticipation that it will.

Now, insert the statistics into the metadata database:

In [42]:
archive_metadata.insert_nightly_stats(simulation_uuid, nightly_stats)

Query the database and look at the results:

In [43]:
archive_metadata.query_nightly_stats(simulation_uuid).head()

,visitseq_uuid,day_obs,value_name,accumulated,count,mean,std,min,p05,q1,median,q3,p95,max
0,dfa5e287-4a42-4443-b79b-fe78584f39f2,2025-06-21,altitude,False,1063,47.069133,12.390993,22.552886,27.013137,37.943459,46.205349,56.211939,67.802678,76.192664
1,dfa5e287-4a42-4443-b79b-fe78584f39f2,2025-06-21,azimuth,False,1063,205.032185,72.475388,0.478060,34.057074,182.036899,211.791488,242.564316,329.137291,358.108652
2,dfa5e287-4a42-4443-b79b-fe78584f39f2,2025-06-21,fiveSigmaDepth,False,1063,23.662358,0.889316,20.601645,21.316253,23.365865,23.862662,24.256195,24.637688,24.836510
3,dfa5e287-4a42-4443-b79b-fe78584f39f2,2025-06-21,observationStartMJD,False,1063,60848.199416,0.142563,60847.950785,60847.977288,60848.076808,60848.199759,60848.321699,60848.422234,60848.445850
4,dfa5e287-4a42-4443-b79b-fe78584f39f2,2025-06-21,rotTelPos,False,1063,-44.062627,9.455674,-78.505054,-64.067769,-47.194495,-43.600436,-38.484309,-31.982576,-4.686711


`value_name` can be any (reasonable) string referring to any scalar quantity: statistics on derived columns or `maf` "stackers" can be added as will.

## Recording the `conda` environment

The contents of the `conda` envirnoment in which a simulation was run can also be included in the metadata database.

Begin by getting the `conda` environment in local python variables:

In [44]:
env_hash, env_json = vseqarchive.compute_conda_env()

Update the `simulations` table with a reference to the environment. This does not by itself save the content of the environment, just a hash of its representation. If you stop after this, you can identify which simulations were made with the same environment, but not query what packages were actually in the environment.

In [45]:
archive_metadata.update_visitseq_metadata(simulation_uuid, 'conda_env_sha256', env_hash)

We can, though, actually store the contents of the environment:

In [46]:
archive_metadata.record_conda_env(env_hash, env_json)

b'\x8eM\r\x02\xe4\x85\xc2\xd8\x84?\xf6?\xc5K\xfb\tw\xcbZ~\xc3\x85\xe3\xf5Y\xa2\\\xfd\xbe\t\xf2^'

In [47]:
archive_metadata.get_visitseq_metadata(simulation_uuid, 'simulations_extra')

sim_creation_day_obs                                           2025-09-23
daily_id                                                                1
visitseq_uuid                        dfa5e287-4a42-4443-b79b-fe78584f39f2
visitseq_label                                  Sample simulated visits 1
visitseq_url            file:///var/folders/_x/wlrrv2nn0mj2y1jdkmphj2f...
telescope                                                         simonyi
first_day_obs                                                  2025-06-21
last_day_obs                                                   2025-06-24
creation_time                            2025-09-23 21:27:51.691148+00:00
scheduler_version                                                  3.15.0
config_url                                                           None
sim_runner_kwargs       {'verbose': True, 'filename': '/var/folders/_x...
conda_env_sha256        [b'\x8e', b'M', b'\r', b'\x02', b'\xe4', b'\x8...
parent_visitseq_uuid                 4

You can use `VisitSequenceArchiveMetadata.query` and join `simulations` with `conda_env` to get the full `json` representation of the conda environment, or use postgresql json operators to get specific information on packages specified in that json. Alternately, you can query the `simulation_packages` view to get data on specific packages associated with specific simulations:

In [48]:
archive_metadata.query(f"""
    SELECT visitseq_uuid, encode(conda_env_hash, 'hex') AS conda_env, package_name, package_version, platform
    FROM simulation_packages
    WHERE visitseq_uuid='{simulation_uuid}'
          AND package_name='numpy'
    """
)

[(UUID('dfa5e287-4a42-4443-b79b-fe78584f39f2'),
  '8e4d0d02e485c2d8843ff63fc54bfb0977cb5a7ec385e3f559a25cfdbe09f25e',
  'numpy',
  '2.2.4',
  'osx-arm64')]

Note the use of `encode(conda_env_hash, 'hex')` to return the a hes representation of the hash of the conda environment instead of an array of bytes, the native PostgreSQL column type.

## Mixed simulations

In the above sections, we recorded sequences of visits queried from consdb, and visits simulated after that.
Often, though, we will want a sequence of visits that include both the queried visits and the newly simulated ones.
The metadata for these can be tracked in the `mixedvisitseq` table in the metadata database.

First, make a `pd.DataFrame` with all the visits:

In [49]:
mixed_visits = pd.concat([consdb_visits, simulated_visits])

Make an entry for this mixed visit sequence in the metadata database:

In [50]:
mixed_uuid = archive_metadata.record_mixed_metadata(
    mixed_visits,
    label="consdb query followed by three more simulated nights",
    last_early_day_obs=np.max(consdb_visits.day_obs),
    first_late_day_obs=Time(sim_start_mjd, format='mjd').datetime.date(),
    early_parent_uuid=sample_consdb_uuid,
    late_parent_uuid=simulation_uuid,
    first_day_obs=np.min(consdb_visits.day_obs),
    last_day_obs=Time(sim_start_mjd + sim_runner_kwargs['sim_duration'], format='mjd').datetime.date()
)

In [51]:
archive_metadata.get_visitseq_metadata(mixed_uuid, 'mixedvisitseq')

visitseq_uuid                      87934b89-650f-4732-90a6-b570140d2373
visitseq_sha256       [b'P', b'=', b'!', b'.', b'\x92', b'\xb4', b'\...
visitseq_label        consdb query followed by three more simulated ...
visitseq_url                                                       None
telescope                                                       simonyi
first_day_obs                                                2025-06-20
last_day_obs                                                 2025-06-24
creation_time                          2025-09-23 21:27:54.491041+00:00
last_early_day_obs                                           2025-06-20
first_late_day_obs                                           2025-06-21
early_parent_uuid                  46d13ef4-9723-48bc-9947-600dd6e81ed1
late_parent_uuid                   dfa5e287-4a42-4443-b79b-fe78584f39f2
Name: 0, dtype: object

## Finding simulations

Directly querying the `simulations` table or `simulations_extra` view with `VisitSequenceArchiveMetadata.query` provides maximum flexibility in specifying exactly what is to be returned, but can be a bit cumbersome.

The `VisitSequenceArchiveMetadata.sims_on_nights` method provides a quick way to query the metadata simulations based on common search parameters, and can be used to narrow down possible simulations of interest to a modest number. Further filtering can then be performed on the returned `pandas.DataFrame`.

The `first_day_obs` and `last_day_obs` specify the date range that must be included in simulations returned, and the `tags` is a list of all tags that must be present. The `max_simulation_age` can be used to limit the simulations returned to only recent ones.

As with other queries based on `day_obs`, the values may be specified as integers, ISO strings, or python `datetime.date` objects.

In [52]:
matching_simulations = archive_metadata.sims_on_nights(
    first_day_obs='2025-06-22',
    last_day_obs='2025-06-23',
    tags=[],
    telescope='simonyi',
    max_simulation_age=100
)
matching_simulations[['sim_creation_day_obs', 'daily_id', 'visitseq_uuid', 'visitseq_label', 'creation_time', 'tags']]

,sim_creation_day_obs,daily_id,visitseq_uuid,visitseq_label,creation_time,tags
0,2025-09-23,1,dfa5e287-4a42-4443-b79b-fe78584f39f2,Sample simulated visits 1,2025-09-23 21:27:51.691148+00:00,"[example, other_tag]"


Note the `sim_creation_day_abs` and `day_id` pair: thin pair provides a shorter alternative to using `visitseq_uuid` as a unique identifier (but is not guaranteed to be unique across different metadata databases), and is also useful for backwards-compatibility with the prototype simulation archive.

## Stop our temporary postgresql database and clean up our temporary directory

In [53]:
md_database.stop()
demo_dir.cleanup()